# [TICKER] — [Company Name]
> **Amaral Asset Management** | Deep Value + Event-Driven | `[DATE]`

---

## Executive Summary

> *One paragraph. Thesis in sentence 1. Variant perception in sentence 2. Catalyst in sentence 3. Position structure in sentence 4.*

| Parameter | Detail |
|---|---|
| Ticker | |
| Sector / Industry | |
| Market Cap | |
| Current Price | |
| Price Target (base) | |
| Bear Case NAV | |
| Upside (base) | |
| Downside (bear) | |
| Risk / Reward | |
| Catalyst | |
| Catalyst Timeline | |
| Position Type | |
| Conviction Tier | |


In [ ]:
# Setup — run this cell first
import sys, os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from screening.yfinance_pull import get_key_metrics, get_price_history, get_financials
from screening.edgar_fetch  import get_filings, get_revenue_history
from models.dcf_model       import DCFModel
from models.comps_analysis  import build_comps_table, print_comps
from models.scenario_model  import ScenarioModel
from models.options_payoff  import OptionsPayoff
from utils.plot_utils       import set_aam_style, revenue_chart, margin_trend_chart
from utils.config           import COLORS

set_aam_style()

# ── CHANGE THESE ──────────────────────────────────────────────
TICKER  = 'ACVA'   # Target ticker
PEERS   = []       # Peer tickers for comps
# ──────────────────────────────────────────────────────────────

print(f'Pulling data for {TICKER}...')

## 1. Business Description & Competitive Position

In [ ]:
metrics = get_key_metrics(TICKER)
print(f"Company:  {metrics['company_name']}")
print(f"Sector:   {metrics['sector']} / {metrics['industry']}")
print(f"Mkt Cap:  ${metrics['market_cap_m']:,.0f}M")
print(f"EV:       ${metrics['enterprise_value_m']:,.0f}M")
print(f"Price:    ${metrics['price']}")
print(f"52w Hi:   ${metrics['52w_high']} | Lo: ${metrics['52w_low']}")
print(f"% of 52w: {metrics['price_to_52w_high']*100:.0f}%")
print()
print('Business description:')
print(metrics['description'])

*[Write 2-3 paragraphs covering:]*
*- What the company does and how it makes money*
*- Competitive position — moat, market share, key competitors*
*- Management quality and capital allocation track record*

## 2. Variant Perception — Why Is the Market Wrong?

*[This is the most important section. Answer:]*
*- What does the consensus believe?*
*- Why is that belief incorrect or incomplete?*
*- What do you see that others don't?*
*- What is the market's mispricing mechanism (neglect, fear, complexity, categorization error)?*

## 3. Financial Analysis

In [ ]:
# Key trading metrics
print('=== Valuation Metrics ===')
print(f"EV/EBITDA:     {metrics['ev_ebitda']}x")
print(f"Forward P/E:   {metrics['pe_forward']}x")
print(f"P/B:           {metrics['pb_ratio']}x")
print(f"FCF yield:     {100/metrics['pe_forward']:.1f}% (approx)" if metrics['pe_forward'] else '')
print()
print('=== Profitability ===')
print(f"Gross margin:  {metrics['gross_margin_pct']}%")
print(f"EBITDA margin: {metrics['ebitda_margin_pct']}%")
print(f"Net margin:    {metrics['net_margin_pct']}%")
print(f"ROE:           {metrics['roe_pct']}%")
print(f"ROA:           {metrics['roa_pct']}%")
print()
print('=== Growth ===')
print(f"Revenue YoY:   {metrics['rev_growth_yoy']}%")
print(f"Earnings YoY:  {metrics['earnings_growth_yoy']}%")
print()
print('=== Balance Sheet ===')
print(f"Debt/Equity:   {metrics['debt_to_equity']}x")
print(f"Current ratio: {metrics['current_ratio']}x")

In [ ]:
# Revenue history from EDGAR
try:
    rev_history = get_revenue_history(TICKER)
    print('Revenue history (from EDGAR):')
    print(rev_history.to_string(index=False))
    
    years = rev_history['fiscal_year_end'].tolist()
    revs  = rev_history['revenue_m'].tolist()
    if len(revs) >= 2:
        fig = revenue_chart(years[-5:], revs[-5:],
                            title=f'{TICKER} — Annual Revenue ($M)')
        plt.show()
except Exception as e:
    print(f'Could not pull EDGAR revenue: {e}')
    print('Enter revenue manually below.')

In [ ]:
# Price history chart
prices = get_price_history(TICKER, years=3)
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(prices.index, prices['Close'], color=COLORS['navy'], linewidth=1.5)
ax.set_title(f'{TICKER} — 3-Year Price History', fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x:.0f}'))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Valuation

In [ ]:
# ── EDIT THESE INPUTS ─────────────────────────────────────────
BASE_REVENUE_TTM   = 485    # $M TTM revenue
NET_DEBT           = -120   # $M (negative = net cash)
SHARES_OUTSTANDING = 155    # M shares diluted
CURRENT_PRICE      = metrics['price']
# ──────────────────────────────────────────────────────────────

# DCF Model
dcf = DCFModel(
    ticker=TICKER,
    base_revenue_ttm=BASE_REVENUE_TTM,
    net_debt=NET_DEBT,
    shares_outstanding=SHARES_OUTSTANDING,
    current_price=CURRENT_PRICE,
)
dcf.run()
dcf.summary()
dcf.sensitivity_table('Base')

In [ ]:
# Scenario Analysis
sm = ScenarioModel(
    ticker=TICKER,
    base_revenue_ttm=BASE_REVENUE_TTM,
    net_debt=NET_DEBT,
    shares_outstanding=SHARES_OUTSTANDING,
    current_price=CURRENT_PRICE,
    projection_years=3,
)
sm.run()
sm.summary()

In [ ]:
# Comparable Company Analysis
if PEERS:
    comps = build_comps_table(
        target=TICKER,
        peers=PEERS,
        target_ebitda_m=BASE_REVENUE_TTM * 0.15,  # adjust
        target_revenue_m=BASE_REVENUE_TTM,
        target_net_debt_m=NET_DEBT,
        target_shares_m=SHARES_OUTSTANDING,
    )
    print_comps(comps)
else:
    print('Add peer tickers to PEERS list at the top to run comps.')

## 5. Catalysts

| Catalyst | Timeline | Probability | Impact |
|---|---|---|---|
| [Primary catalyst] | [Q/Year] | High / Med / Low | Re-rating / Earnings |
| [Secondary catalyst] | [Q/Year] | High / Med / Low | Re-rating / Earnings |

*[Describe each catalyst in 2-3 sentences. Be specific about what needs to happen and why it will close the gap between price and intrinsic value.]*

## 6. Key Risks & Mitigants

| Risk | Severity | Mitigant |
|---|---|---|
| [Risk 1] | High / Med / Low | [How you've accounted for it] |
| [Risk 2] | High / Med / Low | [How you've accounted for it] |
| [Risk 3] | High / Med / Low | [How you've accounted for it] |

**Thesis breakers** — *what would make you exit immediately:*
- [Breaker 1]
- [Breaker 2]

## 7. Position Structure

| Parameter | Detail |
|---|---|
| Instrument | Equity / Long Call / LEAPS / Bull Spread |
| Conviction Tier | Tier 1 / 2 / 3 |
| Target Size | % of portfolio |
| Entry Price | $ |
| Price Target | $ (base case) |
| Bear Case NAV | $ |
| Stop-Loss Trigger | 40% drawdown → thesis re-evaluation |
| Time Horizon | months |
| Expected Catalyst Date | Q/Year |

In [ ]:
# Options payoff (if using options — delete if straight equity)
op = OptionsPayoff(current_price=CURRENT_PRICE, ticker=TICKER)

# Example: LEAPS
# fig, _ = op.plot_leaps(strike=20, premium=3.50,
#                         expiry='Jan 2027', price_target=28.0)

# Example: Bull call spread
# fig, _ = op.plot_bull_spread(K_long=18, K_short=24,
#                               p_long=2.20, p_short=0.80,
#                               expiry='Jun 2025')
# plt.show()

print('Uncomment the relevant options structure above to generate payoff diagram.')

## 8. ESG & Governance

*[Brief notes on:]*
*- Board composition and independence*
*- Insider ownership (% held by management)*
*- Capital allocation history (buybacks, dividends, M&A)*
*- Any ESG flags relevant to the thesis*

---
*Amaral Asset Management | Research completed: [DATE] | AI-assisted analysis verified against primary sources*